# Graphe des commandes livrées
Chaque sommet représente un client ou un restaurant. Une arête agrège le volume et le chiffre d’affaires d’un couple client–restaurant à partir de Silver.

In [ ]:
import os
from pathlib import Path
from tempfile import TemporaryDirectory
from pyspark.sql import functions as F
from src.spark_session import create_spark_session

local_data = os.getenv("FASTFOOD_LOCAL_DATA")
silver_path = (Path(local_data).resolve().parent / "silver" / "orders_history").as_uri() if local_data else "s3a://lake/silver/orders_history/"
spark = create_spark_session("FastFood-GraphFrames", enable_graphframes=True, enable_s3=not local_data)
from graphframes import GraphFrame
from graphframes.lib import AggregateMessages as AM

orders = spark.read.parquet(silver_path).filter("status = 'DELIVERED'")
assert orders.filter("customer_id IS NULL OR restaurant_id IS NULL").count() == 0
clients = orders.select(F.concat(F.lit("client:"), "customer_id").alias("id")).distinct().withColumn("kind", F.lit("client"))
restaurants = (orders.select(F.concat(F.lit("restaurant:"), "restaurant_id").alias("id"), "city", "category")
               .dropDuplicates(["id"]).withColumn("kind", F.lit("restaurant")))
vertices = clients.unionByName(restaurants, allowMissingColumns=True).cache()
edges = (orders.withColumn("src", F.concat(F.lit("client:"), "customer_id"))
         .withColumn("dst", F.concat(F.lit("restaurant:"), "restaurant_id"))
         .groupBy("src", "dst").agg(F.count("*").alias("orders"), F.sum("amount_mad").alias("revenue_mad"))
         .cache())
graph = GraphFrame(vertices, edges)
print("Sommets:", vertices.count(), "| Arêtes:", edges.count())
assert edges.agg(F.sum("orders")).first()[0] == orders.count()
print("Top 10 restaurants par clients distincts")
graph.inDegrees.orderBy(F.desc("inDegree"), "id").show(10, truncate=False)
print("Top 10 clients par restaurants distincts")
graph.outDegrees.orderBy(F.desc("outDegree"), "id").show(10, truncate=False)

In [ ]:
ranked = graph.pageRank(resetProbability=0.15, maxIter=10).vertices
print("Top 10 PageRank restaurants")
(ranked.filter("kind = 'restaurant'").select("id", "pagerank")
 .orderBy(F.desc("pagerank"), "id").show(10, truncate=False))
print("Top 5 restaurants par nombre de commandes")
(edges.groupBy("dst").agg(F.sum("orders").alias("orders"))
 .orderBy(F.desc("orders"), "dst").show(5, truncate=False))

with TemporaryDirectory() as local_checkpoint:
    spark.sparkContext.setCheckpointDir(
        Path(local_checkpoint).as_uri() if local_data else "s3a://lake/checkpoints/graphframes/"
    )
    sizes = graph.connectedComponents(algorithm="graphx").groupBy("component").count()
    stats = sizes.agg(F.count("*").alias("components"), F.max("count").alias("largest")).first()
    print(f"Composantes connexes : {stats.components} | Taille maximale : {stats.largest}")

In [ ]:
motifs = (graph.find("(c)-[e1]->(r1); (c)-[e2]->(r2)")
          .filter("r1.id < r2.id AND r1.category = r2.category"))
print("Clients ayant fréquenté deux restaurants de la même catégorie")
motifs.select("c.id", "r1.id", "r2.id", "r1.category").show(5, truncate=False)

client = graph.outDegrees.orderBy(F.desc("outDegree"), "id").first().id
visited = edges.filter(F.col("src") == client).select(F.col("dst").alias("id"))
categories = restaurants.join(visited, "id").select("category").distinct()
recommendations = (restaurants.join(categories, "category").join(visited, "id", "left_anti")
                   .join(graph.inDegrees, "id").orderBy(F.desc("inDegree"), "id"))
print("Recommandation pour", client)
recommendations.select("id", "category", "inDegree").show(1, truncate=False)

messages = graph.aggregateMessages(F.sum(AM.msg).alias("revenue_mad"), sendToDst=AM.edge["revenue_mad"])
expected = edges.groupBy(F.col("dst").alias("id")).agg(F.sum("revenue_mad").alias("revenue_mad"))
assert messages.exceptAll(expected).count() == 0
assert expected.exceptAll(messages).count() == 0
print("CA par restaurant vérifié via aggregateMessages et groupBy")
messages.orderBy(F.desc("revenue_mad"), "id").show(10, truncate=False)
spark.stop()